# CHAPTER 7 反向传播算法：如何高效地求解梯度？

在第五章和第六章的内容当中，我们以梯度下降算法为核心进行讲解。梯度下降算法告诉了我们模型的参数应该怎么去优化，通过对损失函数求梯度，对所有参数进行更新。

但是我们所有的举的例子都是单隐藏层的例子，我们不需要对参数求二阶导甚至更高阶的导数就能完成对参数的优化。但是在多层次的神经网络当中，参数成千上万是相当正常的事情。而且隐藏层数增多以后，对于参数的求导涉及很长的链式法则。

为了提高计算梯度的效率，我们提出了反向传播算法，正是有了这种算法，我们才能够对庞大的神经网络进行训练。

所以，梯度下降和反向传播是相辅相成的，它们共同完成了对参数的优化。

## 7.1 符号约定与前置知识回顾

在正式进入反向传播之前，我们先回顾第二章建立的前向传播符号体系。设网络共有 $L$ 层（输入不算一层，$\mathbf{a}^{(0)} = \mathbf{x}$），第 $l$ 层（$l = 1, 2, \ldots, L$）的前向传播为：

$$
\begin{aligned}
\mathbf{z}^{(l)} &= \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)} \quad &\text{(线性变换)} \\[4pt]
\mathbf{a}^{(l)} &= h\bigl(\mathbf{z}^{(l)}\bigr) \quad &\text{(激活函数)}
\end{aligned}
$$

<img src="../attachment/图7-1.jpg" width="700" style="display: block; margin: 0 auto;" >

其中：
- $\mathbf{W}^{(l)} \in \mathbb{R}^{n_l \times n_{l-1}}$：第 $l$ 层的权重矩阵，$w_{ji}^{(l)}$ 表示从第 $l-1$ 层的第 $i$ 个神经元连接到第 $l$ 层第 $j$ 个神经元的权重
- $\mathbf{b}^{(l)} \in \mathbb{R}^{n_l}$：偏置向量
- $\mathbf{z}^{(l)} \in \mathbb{R}^{n_l}$：激活前的"净输入"（weighted input）
- $\mathbf{a}^{(l)} \in \mathbb{R}^{n_l}$：激活后的输出
- $h(\cdot)$：激活函数（如 ReLU、Sigmoid 等）

对于单个训练样本 $(\mathbf{x}, \mathbf{y})$，记损失函数为 $\mathcal{L}\bigl(\mathbf{a}^{(L)}, \mathbf{y}\bigr)$，简记为 $\mathcal{L}$。我们的目标是：**高效地求出 $\mathcal{L}$ 对每一层 $\mathbf{W}^{(l)}$ 和 $\mathbf{b}^{(l)}$ 的梯度**。

## 7.2 中间变量 $\delta$：反向传播的"关键枢纽"

直接对 $\mathbf{W}^{(l)}$ 求导需要展开从输出层到第 $l$ 层的漫长链式法则，极为繁琐。反向传播的精妙之处在于引入了一个**中间变量**——**误差信号**（error signal）$\boldsymbol{\delta}^{(l)}$：

> $$
> \boxed{\boldsymbol{\delta}^{(l)} \equiv \frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(l)}}}
> $$

这是一个 $n_l \times 1$ 的列向量，第 $j$ 个分量 $\delta_j^{(l)} = \partial\mathcal{L} / \partial z_j^{(l)}$ 表示：第 $l$ 层第 $j$ 个神经元**净输入的一个微小变化**，会给最终损失带来多大的影响。等价地写为梯度记号：

$$
\boldsymbol{\delta}^{(l)} = \nabla_{\mathbf{z}^{(l)}} \mathcal{L}
$$

## 7.3 四个核心方程：以 $\delta$ 为纽带的完整推导

以 $\boldsymbol{\delta}^{(l)}$ 为核心，整个反向传播算法可以凝练为**四个方程**。这些方程的编号（BP1–BP4）源自经典文献，我们逐一推导。

---

### 方程 BP1：输出层误差（Base Case）

> $$
> \boxed{\boldsymbol{\delta}^{(L)} = \nabla_{\mathbf{a}^{(L)}} \mathcal{L} \;\odot\; h'\bigl(\mathbf{z}^{(L)}\bigr)}
> \tag{BP1}
> $$

**推导**：由定义，$\boldsymbol{\delta}^{(L)}$ 是 $\mathcal{L}$ 对 $\mathbf{z}^{(L)}$ 的梯度，写成分量形式的列向量：

$$
\boldsymbol{\delta}^{(L)} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(L)}}
= \begin{pmatrix}
\dfrac{\partial \mathcal{L}}{\partial z_1^{(L)}} \\[6pt]

\dfrac{\partial \mathcal{L}}{\partial z_2^{(L)}} \\[2pt]

\vdots \\[2pt]
\dfrac{\partial \mathcal{L}}{\partial z_{n_L}^{(L)}}

\end{pmatrix}
$$

对第 $j$ 个分量应用链式法则（以 $a_j^{(L)}$ 为中间变量）。由于 $\mathbf{a}^{(L)} = h(\mathbf{z}^{(L)})$ 是逐元素操作，$a_j^{(L)} = h(z_j^{(L)})$ 仅依赖于 $z_j^{(L)}$：

$$
\frac{\partial \mathcal{L}}{\partial z_j^{(L)}}
= \frac{\partial \mathcal{L}}{\partial a_j^{(L)}} \cdot \frac{\partial a_j^{(L)}}{\partial z_j^{(L)}}
= \frac{\partial \mathcal{L}}{\partial a_j^{(L)}} \cdot h'\bigl(z_j^{(L)}\bigr)
$$

将所有分量收集回列向量，两个列向量的逐元素相乘正是 Hadamard 积 $\odot$：

$$
\boldsymbol{\delta}^{(L)}
= \begin{pmatrix}
\dfrac{\partial \mathcal{L}}{\partial a_1^{(L)}} \\[6pt]
\dfrac{\partial \mathcal{L}}{\partial a_2^{(L)}} \\[2pt]
\vdots \\[2pt]
\dfrac{\partial \mathcal{L}}{\partial a_{n_L}^{(L)}}
\end{pmatrix}
\;\odot\;
\begin{pmatrix}
h'(z_1^{(L)}) \\[2pt]
h'(z_2^{(L)}) \\[2pt]
\vdots \\[2pt]
h'(z_{n_L}^{(L)})
\end{pmatrix}
= \nabla_{\mathbf{a}^{(L)}} \mathcal{L} \;\odot\; h'\bigl(\mathbf{z}^{(L)}\bigr)
$$


**BP1 是反向传播的起点**——它是唯一一个"不依赖其他层 $\delta$"的方程。在递归的语境中，BP1 就是**基案（Base Case）**：当 $l = L$ 时，我们直接从损失函数算出 $\boldsymbol{\delta}^{(L)}$，无需递归。

### 方程 BP2：误差的反向传播（Recursive Step）

> $$
> \boxed{\boldsymbol{\delta}^{(l-1)} = \Bigl(\mathbf{W}^{(l)}\Bigr)^{\!T} \boldsymbol{\delta}^{(l)} \;\odot\; h'\bigl(\mathbf{z}^{(l-1)}\bigr)}
> \tag{BP2}
> $$

**这是四个方程中最核心的一个。** BP2 定义了如何从第 $l$ 层的误差 $\boldsymbol{\delta}^{(l)}$ 算出前一层的误差 $\boldsymbol{\delta}^{(l-1)}$——**这就是反向传播中"传播"二字的数学本质，也是递归关系的灵魂所在。** 注意到 BP2 对 $l = L, L-1, \ldots, 2$ 都成立：从 $\boldsymbol{\delta}^{(L)}$（由 BP1 给出）出发，依次取 $l = L$ 得 $\boldsymbol{\delta}^{(L-1)}$，取 $l = L-1$ 得 $\boldsymbol{\delta}^{(L-2)}$，……直至 $\boldsymbol{\delta}^{(1)}$。

<img src="../attachment/图7-2.jpg" width="700" style="display: block; margin: 0 auto;" >

**推导**：从分量出发。由定义 $\delta_j^{(l-1)} = \dfrac{\partial \mathcal{L}}{\partial z_j^{(l-1)}}$。关键在于：第 $l-1$ 层的 $z_j^{(l-1)}$ 会影响第 $l$ 层的**每一个** $z_k^{(l)}$——先经过 $h$ 变成 $a_j^{(l-1)}$，再通过权重 $w_{kj}^{(l)}$ 流向所有 $z_k^{(l)}$。因此以全部 $\{z_k^{(l)}\}_{k=1}^{n_l}$ 为中间变量，应用多元链式法则对 $k$ 求和：

$$
\delta_j^{(l-1)}
= \sum_{k=1}^{n_l} \frac{\partial \mathcal{L}}{\partial z_k^{(l)}} \cdot \frac{\partial z_k^{(l)}}{\partial z_j^{(l-1)}}
= \sum_{k=1}^{n_l} \delta_k^{(l)} \cdot \frac{\partial z_k^{(l)}}{\partial z_j^{(l-1)}}
\tag{1}
$$

现在计算关键偏导数 $\dfrac{\partial z_k^{(l)}}{\partial z_j^{(l-1)}}$。在前向传播中：

$$
z_k^{(l)} = \sum_{i=1}^{n_{l-1}} w_{ki}^{(l)} \, h\bigl(z_i^{(l-1)}\bigr) + b_k^{(l)}
$$

对 $z_j^{(l-1)}$ 求偏导——注意求和中仅 $i = j$ 那一项含有 $z_j^{(l-1)}$（因为 $h$ 是逐元素函数），其余 $i \neq j$ 项全部为零：

$$
\frac{\partial z_k^{(l)}}{\partial z_j^{(l-1)}}
= w_{kj}^{(l)} \cdot h'\bigl(z_j^{(l-1)}\bigr)
\tag{2}
$$

将 $(2)$ 代入 $(1)$：

$$
\delta_j^{(l-1)}
= \sum_{k=1}^{n_l} \delta_k^{(l)} \cdot w_{kj}^{(l)} \cdot h'\bigl(z_j^{(l-1)}\bigr)
$$

$h'(z_j^{(l-1)})$ 不依赖于求和指标 $k$，提出求和号：

$$
\delta_j^{(l-1)}
= \left( \sum_{k=1}^{n_l} w_{kj}^{(l)} \, \delta_k^{(l)} \right)\;\cdot\; h'\bigl(z_j^{(l-1)}\bigr)
\tag{3}
$$

---


**从分量到列向量**。将 $(3)$ 对所有 $j = 1, 2, \ldots, n_{l-1}$ 同时写出，得到完整的列向量 $\boldsymbol{\delta}^{(l-1)}$：

$$
\boldsymbol{\delta}^{(l-1)}
= \begin{pmatrix}
\delta_1^{(l-1)} \\[4pt]
\delta_2^{(l-1)} \\[2pt]
\vdots \\[2pt]
\delta_{n_{l-1}}^{(l-1)}
\end{pmatrix}
= \begin{pmatrix}
\left( \displaystyle\sum_{k=1}^{n_l} w_{k1}^{(l)} \delta_k^{(l)} \right) \cdot h'(z_1^{(l-1)}) \\[12pt]
\left( \displaystyle\sum_{k=1}^{n_l} w_{k2}^{(l)} \delta_k^{(l)} \right) \cdot h'(z_2^{(l-1)}) \\[10pt]
\vdots \\[10pt]
\left( \displaystyle\sum_{k=1}^{n_l} w_{k n_{l-1}}^{(l)} \delta_k^{(l)} \right) \cdot h'(z_{n_{l-1}}^{(l-1)})
\end{pmatrix}
$$

将每行的乘积拆为 Hadamard 积的两个列向量：

$$
= \begin{pmatrix}
\displaystyle\sum_{k=1}^{n_l} w_{k1}^{(l)} \delta_k^{(l)} \\[10pt]
\displaystyle\sum_{k=1}^{n_l} w_{k2}^{(l)} \delta_k^{(l)} \\[8pt]
\vdots \\[8pt]
\displaystyle\sum_{k=1}^{n_l} w_{k n_{l-1}}^{(l)} \delta_k^{(l)}
\end{pmatrix}
\;\odot\;
\begin{pmatrix}
h'(z_1^{(l-1)}) \\[4pt]
h'(z_2^{(l-1)}) \\[2pt]
\vdots \\[2pt]
h'(z_{n_{l-1}}^{(l-1)})
\end{pmatrix}
\tag{4}
$$

---


**从求和到转置矩阵**。关键一步：识别分量 ① 中的求和。展开第一个分量看：

$$
    \sum_{k=1}^{n_l} w_{k1}^{(l)} \delta_k^{(l)}
    = w_{11}^{(l)} \delta_1^{(l)} + w_{21}^{(l)} \delta_2^{(l)} + \cdots + w_{n_l 1}^{(l)} \delta_{n_l}^{(l)}
$$

这是用矩阵的第 1 列（$\mathbf{W}^{(l)}$ 中所有行在第 1 列上的元素）与向量 $\boldsymbol{\delta}^{(l)}$ 做内积。推广到所有 $j$，整个列向量 ① 就是用一个矩阵去乘 $\boldsymbol{\delta}^{(l)}$，该矩阵的**第 $j$ 行第 $k$ 列**恰好是 $w_{kj}^{(l)}$——注意指标顺序！这正说明这个矩阵是 $\mathbf{W}^{(l)}$ 的转置：

$$
    \begin{pmatrix}
    \sum_k w_{k1}^{(l)} \delta_k^{(l)} \\[6pt]
    \sum_k w_{k2}^{(l)} \delta_k^{(l)} \\[4pt]
    \vdots \\[4pt]
    \sum_k w_{k n_{l-1}}^{(l)} \delta_k^{(l)}
    \end{pmatrix}
    =
    \underbrace{\begin{pmatrix}
    w_{11}^{(l)} & w_{21}^{(l)} & \cdots & w_{n_l 1}^{(l)} \\[3pt]
    w_{12}^{(l)} & w_{22}^{(l)} & \cdots & w_{n_l 2}^{(l)} \\[2pt]
    \vdots & \vdots & \ddots & \vdots \\[2pt]
    w_{1 n_{l-1}}^{(l)} & w_{2 n_{l-1}}^{(l)} & \cdots & w_{n_l n_{l-1}}^{(l)}
    \end{pmatrix}}_{(\mathbf{W}^{(l)})^T \;\in\; \mathbb{R}^{n_{l-1} \times n_l}}
    \begin{pmatrix}
    \delta_1^{(l)} \\[4pt]
    \delta_2^{(l)} \\[2pt]
    \vdots \\[2pt]
    \delta_{n_l}^{(l)}
    \end{pmatrix}
    = \Bigl(\mathbf{W}^{(l)}\Bigr)^{\!T} \boldsymbol{\delta}^{(l)}
    \tag{5}
$$

将 $(5)$ 代入 $(4)$，即得 BP2 的最终形式：

$$
    \boxed{\boldsymbol{\delta}^{(l-1)} = \Bigl(\mathbf{W}^{(l)}\Bigr)^{\!T} \boldsymbol{\delta}^{(l)} \;\odot\; h'\bigl(\mathbf{z}^{(l-1)}\bigr)}
$$


---

### 深度解读 BP2：它到底在做什么？

把 BP2 拆成两步来理解：

$$
\underbrace{\Bigl(\mathbf{W}^{(l)}\Bigr)^{\!T} \boldsymbol{\delta}^{(l)}}_{\text{Step 1: 误差回传}}
\;\odot\;
\underbrace{h'\bigl(\mathbf{z}^{(l-1)}\bigr)}_{\text{Step 2: 门控修正}}
$$

- **Step 1 — 误差回传**：$\bigl(\mathbf{W}^{(l)}\bigr)^T \boldsymbol{\delta}^{(l)}$ 把第 $l$ 层的误差信号沿着权重矩阵的转置"反向输送"回第 $l-1$ 层。这正是上节推导中从求和 $\sum_k w_{kj}^{(l)} \delta_k^{(l)}$ 识别出 $(\mathbf{W}^{(l)})^T \boldsymbol{\delta}^{(l)}$ 的几何含义——前向传播用 $\mathbf{W}^{(l)}$ 将信息**发散**，反向传播用其**转置**将误差重新**汇聚**。

- **Step 2 — 门控修正**：乘上 $h'(\mathbf{z}^{(l-1)})$ 是对误差信号的一次"筛选"。以 ReLU 为例，$h'(z) = 1$ 当 $z > 0$，否则为 $0$——**在 ReLU 关断的神经元上，误差信号被完全阻断**（这正是"Dying ReLU"现象的数学根源）。以 Sigmoid 为例，$h'(z)$ 在 $|z|$ 很大时趋于 $0$，误差信号衰减，从而导致梯度消失。

### 方程 BP3 & BP4：从误差信号到参数梯度

有了 $\boldsymbol{\delta}^{(l)}$，对参数 $\mathbf{b}^{(l)}$ 和 $\mathbf{W}^{(l)}$ 的梯度几乎唾手可得。

---

#### 方程 BP3：偏置梯度

> $$
> \boxed{\nabla_{\mathbf{b}^{(l)}} \mathcal{L} = \boldsymbol{\delta}^{(l)}}
> \tag{BP3}
> $$

**推导**：写出第 $l$ 层第 $j$ 个神经元的净输入：

$$
z_j^{(l)} = \sum_{i=1}^{n_{l-1}} w_{ji}^{(l)} a_i^{(l-1)} + b_j^{(l)}
$$

对其中的 $b_j^{(l)}$ 求偏导，$\dfrac{\partial z_j^{(l)}}{\partial b_j^{(l)}} = 1$（求和项和偏置无关）。由链式法则：

$$
\frac{\partial \mathcal{L}}{\partial b_j^{(l)}}
= \frac{\partial \mathcal{L}}{\partial z_j^{(l)}} \cdot \frac{\partial z_j^{(l)}}{\partial b_j^{(l)}}
= \delta_j^{(l)} \cdot 1
= \delta_j^{(l)}
$$

把所有 $j$ 的分量收集回列向量，即得 $\nabla_{\mathbf{b}^{(l)}} \mathcal{L} = \boldsymbol{\delta}^{(l)}$。


---

#### 方程 BP4：权重梯度

> $$
> \boxed{\nabla_{\mathbf{W}^{(l)}} \mathcal{L} = \boldsymbol{\delta}^{(l)} \bigl(\mathbf{a}^{(l-1)}\bigr)^T}
> \tag{BP4}
> $$

**推导**：由 $z_j^{(l)} = \sum_{i=1}^{n_{l-1}} w_{ji}^{(l)} a_i^{(l-1)} + b_j^{(l)}$，对单个权重 $w_{ji}^{(l)}$ 求偏导：

$$
\frac{\partial \mathcal{L}}{\partial w_{ji}^{(l)}}
= \frac{\partial \mathcal{L}}{\partial z_j^{(l)}} \cdot \frac{\partial z_j^{(l)}}{\partial w_{ji}^{(l)}}
= \delta_j^{(l)} \cdot a_i^{(l-1)}
$$

将所有 $(j, i)$ 对排成 $n_l \times n_{l-1}$ 矩阵：

$$
\nabla_{\mathbf{W}^{(l)}} \mathcal{L}
= \begin{pmatrix}
\delta_1^{(l)} a_1^{(l-1)} & \delta_1^{(l)} a_2^{(l-1)} & \cdots & \delta_1^{(l)} a_{n_{l-1}}^{(l-1)} \\[3pt]
\delta_2^{(l)} a_1^{(l-1)} & \delta_2^{(l)} a_2^{(l-1)} & \cdots & \delta_2^{(l)} a_{n_{l-1}}^{(l-1)} \\[2pt]
\vdots & \vdots & \ddots & \vdots \\[2pt]
\delta_{n_l}^{(l)} a_1^{(l-1)} & \delta_{n_l}^{(l)} a_2^{(l-1)} & \cdots & \delta_{n_l}^{(l)} a_{n_{l-1}}^{(l-1)}
\end{pmatrix}
= \underbrace{\begin{pmatrix} \delta_1^{(l)} \\ \delta_2^{(l)} \\ \vdots \\ \delta_{n_l}^{(l)} \end{pmatrix}}_{n_l \times 1}
\;\cdot\;
\underbrace{\begin{pmatrix} a_1^{(l-1)} & a_2^{(l-1)} & \cdots & a_{n_{l-1}}^{(l-1)} \end{pmatrix}}_{1 \times n_{l-1}}
$$

这正是**外积**（outer product）$\boldsymbol{\delta}^{(l)} (\mathbf{a}^{(l-1)})^T$——列向量 $\boldsymbol{\delta}^{(l)}$ 与行向量 $(\mathbf{a}^{(l-1)})^T$ 相乘，得到 $n_l \times n_{l-1}$ 矩阵，维度与 $\mathbf{W}^{(l)}$ 完全一致。


---

### 反向传播方程组

将四个核心方程汇总为反向传播方程组：

$$
\boxed{
\begin{aligned}
\boldsymbol{\delta}^{(L)} &= \nabla_{\mathbf{a}^{(L)}} \mathcal{L} \;\odot\; h'\bigl(\mathbf{z}^{(L)}\bigr) \\[8pt]
\boldsymbol{\delta}^{(l-1)} &= \bigl(\mathbf{W}^{(l)}\bigr)^{\!T} \boldsymbol{\delta}^{(l)} \;\odot\; h'\bigl(\mathbf{z}^{(l-1)}\bigr) \\[8pt]
\nabla_{\mathbf{b}^{(l)}} \mathcal{L} &= \boldsymbol{\delta}^{(l)} \\[8pt]
\nabla_{\mathbf{W}^{(l)}} \mathcal{L} &= \boldsymbol{\delta}^{(l)} \bigl(\mathbf{a}^{(l-1)}\bigr)^T
\end{aligned}
\qquad
\begin{aligned}
&\text{(BP1) 基案} \\[8pt]
&\text{(BP2) 递归} \\[8pt]
&\text{(BP3) 偏置梯度} \\[8pt]
&\text{(BP4) 权重梯度}
\end{aligned}}
$$

其算法的迭代方式可以大致用下面的流程展示：

$$
\underbrace{\boldsymbol{\delta}^{(L)}}_{\text{BP1}}
\;\xrightarrow{\text{BP2}\; l=L}\;
\underbrace{\boldsymbol{\delta}^{(L-1)}}_{\text{BP3,4}\;\Rightarrow\;\nabla\mathbf{W}^{(L-1)},\nabla\mathbf{b}^{(L-1)}}
\;\xrightarrow{\text{BP2}\; l=L-1}\;
\underbrace{\boldsymbol{\delta}^{(L-2)}}_{\text{BP3,4}\;\Rightarrow\;\nabla\mathbf{W}^{(L-2)},\nabla\mathbf{b}^{(L-2)}}
\;\xrightarrow{\text{BP2}}\;
\cdots
\;\xrightarrow{\text{BP2}\; l=2}\;
\underbrace{\boldsymbol{\delta}^{(1)}}_{\text{BP3,4}\;\Rightarrow\;\nabla\mathbf{W}^{(1)},\nabla\mathbf{b}^{(1)}}
$$

从最后一层L层开始，利用BP2方程不断倒推上一层的误差信号 ${\delta}^{(l-1)}$ ，然后计算本层的权重梯度和偏执梯度，去更新参数。


## 7.4 小结

通过上面的方程组的推导，我们看到了表述反向传播算法的最精炼的模式。如果你仔细观察的话，你可能会联想到它与前向传播之间的关系。我们看到BP2方程中，需要 $\mathbf{z}^{(l-1)}$，在BP4方程中，需要 $\mathbf{a}^{(l-1)}$。也就是说在计算梯度的时候需要每一层的激活前数值 $\mathbf{z}^{(l)}$ 和激活后数值 $\mathbf{a}^{(l)}$ 。

我们在前向传播的时候就必须要计算每一层的这两种数值，如果我们天才般地预见了反向传播的形式，我们就会考虑保留每一层的 $\mathbf{z}^{(l)}$ ，$\mathbf{a}^{(l)}$ 而不是算完以后甩手就把它给扔掉。如果你幸运地保留了这些值，你在返回来算梯度的时候就会感谢自己的

这与现实中是类似的，你需要记录你学习的过程，等到返回头复习的时候就会有迹可寻。但实际上我们总是没有办法预见未来你需要什么，所以为了不后悔，每一步都要脚踏实地、做好记录。